In [2]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np
import importlib
import functions as f
from scintkit.preprocessing.format import temp_formating
from scintkit.services.phase_detrend import detect_sampling_rate
import time
import CONFIG as cf

importlib.reload(f)
importlib.reload(cf)
start = time.time()

# import pqs
dfa = pd.read_parquet(cf.Data_folder1)
dfb = pd.read_parquet(cf.Data_folder2)

print(f"time to read files: {time.time() - start:.3f} seconds")

# filter dfs to contain certain elevation
dfa = dfa[dfa['elev'] > 20].copy()
dfb = dfb[dfb['elev'] > 20].copy()

# fit distance into here?

display(dfa)


time to read files: 1.528 seconds


,leap,cons,svid,elev,azim,snr1,snr2,snr3,pst3,rst3,cph1,cph2,cph3,rng1,rng2,rng3,lon,lat,hei,datetime
1,18,0,12,29,23,43,40,0,0,0,1.178793e+08,9.185398e+07,0.0,2.243167e+07,2.243168e+07,0.0,-359060.78125,-72122.414062,560.80603,2022-10-04 20:00:18.250000
3,18,0,15,36,92,44,42,0,0,0,1.169097e+08,9.109848e+07,0.0,2.224718e+07,2.224719e+07,0.0,-359060.78125,-72122.414062,560.80603,2022-10-04 20:00:18.250000
4,18,0,23,66,297,46,47,0,0,0,1.064099e+08,8.291676e+07,0.0,2.024909e+07,2.024910e+07,0.0,-359060.78125,-72122.414062,560.80603,2022-10-04 20:00:18.250000
6,18,3,19,39,342,48,0,0,0,0,1.217141e+08,0.000000e+00,0.0,2.337388e+07,0.000000e+00,0.0,-359060.78125,-72122.414062,560.80603,2022-10-04 20:00:18.250000
7,18,3,20,73,73,51,0,0,0,0,1.104161e+08,0.000000e+00,0.0,2.120422e+07,0.000000e+00,0.0,-359060.78125,-72122.414062,560.80603,2022-10-04 20:00:18.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11839650,18,2,3,69,284,49,52,0,0,0,1.382457e+08,1.059287e+08,0.0,2.630724e+07,2.630724e+07,0.0,-359061.53125,-72122.351562,559.88501,2022-10-04 23:59:24.953125
11839651,18,2,36,46,93,45,49,0,0,0,1.439516e+08,1.103007e+08,0.0,2.739306e+07,2.739307e+07,0.0,-359061.53125,-72122.351562,559.88501,2022-10-04 23:59:24.953125
11839652,18,0,10,27,168,38,40,0,0,0,1.340101e+08,1.044235e+08,0.0,2.550128e+07,2.550129e+07,0.0,-359061.53125,-72122.351562,559.88501,2022-10-04 23:59:25.015625
11839653,18,0,27,51,242,49,48,0,0,0,1.241874e+08,9.676943e+07,0.0,2.363207e+07,2.363208e+07,0.0,-359061.53125,-72122.351562,559.88501,2022-10-04 23:59:25.015625


In [3]:
# individual sampling rates
dfa = temp_formating(dfa)
dfb = temp_formating(dfb)


display(dfa)

,index,leap,cons,svid,elev,azim,snr1,snr2,snr3,pst3,...,hei,datetime,minbin,prn,sig_1,sig_2,sig_3,freq_1,freq_2,freq_3
0,1,18,GPS,12,29,23,43.0,40.0,NaN,0,...,560.80603,2022-10-04 20:00:18.250000,2022-10-04 20:00:00,G12,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45
1,3,18,GPS,15,36,92,44.0,42.0,NaN,0,...,560.80603,2022-10-04 20:00:18.250000,2022-10-04 20:00:00,G15,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45
2,4,18,GPS,23,66,297,46.0,47.0,NaN,0,...,560.80603,2022-10-04 20:00:18.250000,2022-10-04 20:00:00,G23,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45
3,6,18,BDS,19,39,342,48.0,NaN,NaN,0,...,560.80603,2022-10-04 20:00:18.250000,2022-10-04 20:00:00,C19,BDS_B1I,BDS_B2I,BDS_B3I,1561.098,1207.14,1268.52
4,7,18,BDS,20,73,73,51.0,NaN,NaN,0,...,560.80603,2022-10-04 20:00:18.250000,2022-10-04 20:00:00,C20,BDS_B1I,BDS_B2I,BDS_B3I,1561.098,1207.14,1268.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6752776,11839650,18,GAL,3,69,284,49.0,52.0,NaN,0,...,559.88501,2022-10-04 23:59:24.953125,2022-10-04 23:59:00,E03,GAL_L1BC,GAL_E5b,GAL_E5b,1575.420,1207.14,1207.14
6752777,11839651,18,GAL,36,46,93,45.0,49.0,NaN,0,...,559.88501,2022-10-04 23:59:24.953125,2022-10-04 23:59:00,E36,GAL_L1BC,GAL_E5b,GAL_E5b,1575.420,1207.14,1207.14
6752778,11839652,18,GPS,10,27,168,38.0,40.0,NaN,0,...,559.88501,2022-10-04 23:59:25.015625,2022-10-04 23:59:00,G10,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45
6752779,11839653,18,GPS,27,51,242,49.0,48.0,NaN,0,...,559.88501,2022-10-04 23:59:25.015625,2022-10-04 23:59:00,G27,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45


In [4]:

samp_ra = detect_sampling_rate(dfa)
samp_rb = detect_sampling_rate(dfb)

# print(samp_ra)
# print(samp_rb)

dt = 1 / samp_ra

# merge 2 receiver dfs
merged = dfa.merge(dfb, on=["datetime", "svid", "cons"], suffixes=("_A", "_B"))
merged['snr_diff'] = abs(merged['snr1_A'] - merged['snr1_B'])

display(merged)
print(merged.columns)

,index_A,leap_A,cons,svid,elev_A,azim_A,snr1_A,snr2_A,snr3_A,pst3_A,...,hei_B,minbin_B,prn_B,sig_1_B,sig_2_B,sig_3_B,freq_1_B,freq_2_B,freq_3_B,snr_diff
0,1,18,GPS,12,29,23,43.0,40.0,NaN,0,...,543.103027,2022-10-04 20:00:00,G12,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45,0.0
1,3,18,GPS,15,36,92,44.0,42.0,NaN,0,...,543.103027,2022-10-04 20:00:00,G15,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45,2.0
2,4,18,GPS,23,66,297,46.0,47.0,NaN,0,...,543.103027,2022-10-04 20:00:00,G23,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45,2.0
3,6,18,BDS,19,39,342,48.0,NaN,NaN,0,...,543.103027,2022-10-04 20:00:00,C19,BDS_B1I,BDS_B2I,BDS_B3I,1561.098,1207.14,1268.52,1.0
4,7,18,BDS,20,73,73,51.0,NaN,NaN,0,...,543.103027,2022-10-04 20:00:00,C20,BDS_B1I,BDS_B2I,BDS_B3I,1561.098,1207.14,1268.52,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3649432,11839641,18,GPS,32,72,173,51.0,49.0,NaN,0,...,543.080017,2022-10-04 23:59:00,G32,GPS_L1CA,GPS_L2C,GPS_L5,1575.420,1227.60,1176.45,1.0
3649433,11839644,18,GAL,34,46,175,49.0,49.0,NaN,0,...,543.080017,2022-10-04 23:59:00,E34,GAL_L1BC,GAL_E5b,GAL_E5b,1575.420,1207.14,1207.14,2.0
3649434,11839649,18,GAL,5,48,159,48.0,52.0,NaN,0,...,543.080017,2022-10-04 23:59:00,E05,GAL_L1BC,GAL_E5b,GAL_E5b,1575.420,1207.14,1207.14,4.0
3649435,11839650,18,GAL,3,69,284,49.0,52.0,NaN,0,...,543.080017,2022-10-04 23:59:00,E03,GAL_L1BC,GAL_E5b,GAL_E5b,1575.420,1207.14,1207.14,0.0


Index(['index_A', 'leap_A', 'cons', 'svid', 'elev_A', 'azim_A', 'snr1_A',
       'snr2_A', 'snr3_A', 'pst3_A', 'rst3_A', 'cph1_A', 'cph2_A', 'cph3_A',
       'rng1_A', 'rng2_A', 'rng3_A', 'lon_A', 'lat_A', 'hei_A', 'datetime',
       'minbin_A', 'prn_A', 'sig_1_A', 'sig_2_A', 'sig_3_A', 'freq_1_A',
       'freq_2_A', 'freq_3_A', 'index_B', 'leap_B', 'elev_B', 'azim_B',
       'snr1_B', 'snr2_B', 'snr3_B', 'pst3_B', 'rst3_B', 'cph1_B', 'cph2_B',
       'cph3_B', 'rng1_B', 'rng2_B', 'rng3_B', 'lon_B', 'lat_B', 'hei_B',
       'minbin_B', 'prn_B', 'sig_1_B', 'sig_2_B', 'sig_3_B', 'freq_1_B',
       'freq_2_B', 'freq_3_B', 'snr_diff'],
      dtype='object')


In [5]:
# merged = temp_formating(merged)

# display(merged)
# print(merged.columns)

#remove this cuz unecessary

In [6]:

print(f"time to create new time: {time.time() - start:.3f} seconds")

thresh = cf.thresh  # threshold for s4 scintillation measurement


#### start cross corr file creation

rstart = time.time()

scint = []

sat_groups = merged.groupby(['svid', 'cons'])

for (svid, cons), sat_group in sat_groups:

    # group the data into different satellites
    sat_group = f.datetime_to_seconds(sat_group)

    min_groups = sat_group.groupby(sat_group['datetime'].dt.floor('min'))

    # with the chosen satellite for this iteration
    # find s4 and then determine scintillation
    for min, group in min_groups:
        
        f.handle_nan(group, cf.nan_method)
        #add nan processing
        if len(group) < 10:
            continue

        sig_lina = f.db2lin(group['snr1_A'])
        s4a = np.std(sig_lina) / np.mean(sig_lina)

        sig_linb = f.db2lin(group['snr1_B'])
        s4b = np.std(sig_linb) / np.mean(sig_linb)

        if s4a > thresh or s4b > thresh:

            # check threshold of scintillation and compute correlation
            correlation, lag_b, cor_norm, lag_norm = f.cross_correlation(group["snr1_A"], group["snr1_B"])
            autoA_max, autoA_lagb, autoA_cor, autoA_lags = f.cross_correlation(group['snr1_A'], group['snr1_A'])
            autoB_max, autoB_lagb, autoB_cor, autoB_lags = f.cross_correlation(group['snr1_B'], group['snr1_B'])

            # dt = group['time_sec'].diff().median()
            time_delay = lag_b * dt

            scint.append({
                'minute': min,
                'prn' : group['prn_B'].iloc[0],
                's4A': s4a,
                's4B': s4b,
                

                #adding elev and azim
                'elev' : group['elev_A'].mean(),
                'azim' : group['azim_A'].mean(),
                
                #adding location, using the location at the start of each minute not the mean, can be changed
                'rAloc' : (group['lat_A'].iloc[0]/10000, group['lon_A'].iloc[0]/10000, group['hei_A'].mean()/1000),
                'rBloc' : (group['lat_B'].iloc[0]/10000, group['lon_B'].iloc[0]/10000, group['hei_B'].mean()/1000),

                'auto_cor_A' : autoA_cor,
                'auto_cor_Amax' : autoA_max,
                'auto_cor_B' : autoB_cor,
                'auto_cor_Bmax' : autoB_max,


                'corr_norm': cor_norm,
                'lag_norm': lag_norm,
                'max_corr': correlation,
                #'best_lag': lag_b,
                'time_delay': time_delay
            })

# create dataframe storing all scintillation events
cross_cor = pd.DataFrame(scint)
display(cross_cor)

print(cross_cor['prn'].unique())

time to create new time: 30.104 seconds


,minute,prn,s4A,s4B,elev,azim,rAloc,rBloc,auto_cor_A,auto_cor_Amax,auto_cor_B,auto_cor_Bmax,corr_norm,lag_norm,max_corr,time_delay
0,2022-10-04 22:22:00,E02,0.248260,0.246203,32.000000,306.131250,"(-7.2121882, -35.90611, 0.55649865)","(-7.212652, -35.9073, 0.5425475)","[0.0007473841554558992, 0.0014947683109117983,...",1.0,"[7.453151618398635e-05, 0.000489778534923339, ...",1.0,"[0.00023601625850355374, 0.0015509639844519247...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.724705,0.30
1,2022-10-04 22:23:00,E02,0.258300,0.282083,32.000000,307.000000,"(-7.2121882, -35.906105, 0.5563686)","(-7.212652, -35.907295, 0.542493)","[-0.0002862061228668591, -0.000572412245733718...",1.0,"[5.653192470854709e-08, 1.1306384941709418e-07...",1.0,"[-1.7930009852308093e-05, -3.5860019704616186e...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.645983,0.30
2,2022-10-04 22:24:00,E02,0.356450,0.339789,31.639413,307.360587,"(-7.2121873, -35.906105, 0.55611974)","(-7.212652, -35.907295, 0.54223895)","[-6.744767325199305e-05, -0.000134895346503986...",1.0,"[1.4328560169239606e-05, 2.8657120338479213e-0...",1.0,"[-0.0006721788745185038, -0.001344357749037007...","[-476, -475, -474, -473, -472, -471, -470, -46...",0.811655,0.30
3,2022-10-04 22:25:00,E02,0.316135,0.314274,31.000000,308.000000,"(-7.2121873, -35.906105, 0.55578876)","(-7.212652, -35.907295, 0.5419835)","[-0.00026176338703215663, -0.00052352677406431...",1.0,"[-0.00020526582306656593, -0.00030202107887327...",1.0,"[4.9965221128560714e-05, 9.993044225712143e-05...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.744665,0.30
4,2022-10-04 22:26:00,E02,0.345028,0.325862,31.000000,308.000000,"(-7.2121882, -35.906105, 0.5553776)","(-7.212654, -35.907295, 0.5417687)","[-0.0003301509261376589, -0.000660301852275317...",1.0,"[-0.00012863502852089365, 0.000664294326391474...",1.0,"[-0.0001734960252106696, -0.000346992050421339...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.686357,0.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,2022-10-04 23:37:00,S136,0.332755,0.340081,42.000000,82.000000,"(-7.2122126, -35.906174, 0.55261135)","(-7.2126813, -35.907364, 0.53728354)","[-8.878722740604429e-05, -0.000177574454812088...",1.0,"[-0.0015268451462488895, -0.003053690292497779...",1.0,"[-0.0002986801244202465, -0.000597360248840493...","[-949, -948, -947, -946, -945, -944, -943, -94...",0.862867,0.90
1531,2022-10-04 23:38:00,S136,0.311478,0.274253,42.000000,82.000000,"(-7.2122173, -35.906174, 0.5526083)","(-7.2126837, -35.907364, 0.5371062)","[0.0005966432587921004, 0.0011932865175842008,...",1.0,"[-0.0020429527221507743, -0.004085905444301549...",1.0,"[-0.001273931181367274, -0.002547862362734548,...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.798184,0.90
1532,2022-10-04 23:39:00,S136,0.263936,0.258914,42.000000,82.000000,"(-7.2122197, -35.906174, 0.55342394)","(-7.2126837, -35.907364, 0.53767467)","[-0.0007197609850313849, -0.001439521970062769...",1.0,"[0.0010492216341598936, 0.002098443268319787, ...",1.0,"[-0.000405569010462156, -0.000811138020924312,...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.765527,0.80
1533,2022-10-04 23:40:00,S136,0.212937,0.179114,42.000000,82.000000,"(-7.2122235, -35.906166, 0.55444056)","(-7.2126865, -35.90736, 0.5384905)","[-0.001322385975152172, -0.002644771950304344,...",1.0,"[-0.0017284283257443301, -0.001142642993588310...",1.0,"[-0.0017825553229625494, -0.003565110645925099...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.589147,1.15


['E02' 'E03' 'R04' 'E05' 'R05' 'R06' 'R07' 'R08' 'E09' 'R09' 'G10' 'G12'
 'E14' 'G15' 'G16' 'G18' 'C19' 'R19' 'C20' 'R20' 'R21' 'C22' 'G22' 'G23'
 'E24' 'E25' 'G25' 'G26' 'G27' 'C29' 'G29' 'C30' 'G31' 'C32' 'G32' 'E34'
 'C35' 'C36' 'S136']


In [7]:

cross_cor['distance (km)'] = cross_cor.apply(lambda row: f.calc_dist(row['rAloc'], row['rBloc']), axis = 1)



display(cross_cor)


print(cross_cor.columns)

,minute,prn,s4A,s4B,elev,azim,rAloc,rBloc,auto_cor_A,auto_cor_Amax,auto_cor_B,auto_cor_Bmax,corr_norm,lag_norm,max_corr,time_delay,distance (km)
0,2022-10-04 22:22:00,E02,0.248260,0.246203,32.000000,306.131250,"(-7.2121882, -35.90611, 0.55649865)","(-7.212652, -35.9073, 0.5425475)","[0.0007473841554558992, 0.0014947683109117983,...",1.0,"[7.453151618398635e-05, 0.000489778534923339, ...",1.0,"[0.00023601625850355374, 0.0015509639844519247...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.724705,0.30,0.141599
1,2022-10-04 22:23:00,E02,0.258300,0.282083,32.000000,307.000000,"(-7.2121882, -35.906105, 0.5563686)","(-7.212652, -35.907295, 0.542493)","[-0.0002862061228668591, -0.000572412245733718...",1.0,"[5.653192470854709e-08, 1.1306384941709418e-07...",1.0,"[-1.7930009852308093e-05, -3.5860019704616186e...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.645983,0.30,0.141591
2,2022-10-04 22:24:00,E02,0.356450,0.339789,31.639413,307.360587,"(-7.2121873, -35.906105, 0.55611974)","(-7.212652, -35.907295, 0.54223895)","[-6.744767325199305e-05, -0.000134895346503986...",1.0,"[1.4328560169239606e-05, 2.8657120338479213e-0...",1.0,"[-0.0006721788745185038, -0.001344357749037007...","[-476, -475, -474, -473, -472, -471, -470, -46...",0.811655,0.30,0.141626
3,2022-10-04 22:25:00,E02,0.316135,0.314274,31.000000,308.000000,"(-7.2121873, -35.906105, 0.55578876)","(-7.212652, -35.907295, 0.5419835)","[-0.00026176338703215663, -0.00052352677406431...",1.0,"[-0.00020526582306656593, -0.00030202107887327...",1.0,"[4.9965221128560714e-05, 9.993044225712143e-05...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.744665,0.30,0.141619
4,2022-10-04 22:26:00,E02,0.345028,0.325862,31.000000,308.000000,"(-7.2121882, -35.906105, 0.5553776)","(-7.212654, -35.907295, 0.5417687)","[-0.0003301509261376589, -0.000660301852275317...",1.0,"[-0.00012863502852089365, 0.000664294326391474...",1.0,"[-0.0001734960252106696, -0.000346992050421339...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.686357,0.35,0.141635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,2022-10-04 23:37:00,S136,0.332755,0.340081,42.000000,82.000000,"(-7.2122126, -35.906174, 0.55261135)","(-7.2126813, -35.907364, 0.53728354)","[-8.878722740604429e-05, -0.000177574454812088...",1.0,"[-0.0015268451462488895, -0.003053690292497779...",1.0,"[-0.0002986801244202465, -0.000597360248840493...","[-949, -948, -947, -946, -945, -944, -943, -94...",0.862867,0.90,0.141914
1531,2022-10-04 23:38:00,S136,0.311478,0.274253,42.000000,82.000000,"(-7.2122173, -35.906174, 0.5526083)","(-7.2126837, -35.907364, 0.5371062)","[0.0005966432587921004, 0.0011932865175842008,...",1.0,"[-0.0020429527221507743, -0.004085905444301549...",1.0,"[-0.001273931181367274, -0.002547862362734548,...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.798184,0.90,0.141864
1532,2022-10-04 23:39:00,S136,0.263936,0.258914,42.000000,82.000000,"(-7.2122197, -35.906174, 0.55342394)","(-7.2126837, -35.907364, 0.53767467)","[-0.0007197609850313849, -0.001439521970062769...",1.0,"[0.0010492216341598936, 0.002098443268319787, ...",1.0,"[-0.000405569010462156, -0.000811138020924312,...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.765527,0.80,0.141787
1533,2022-10-04 23:40:00,S136,0.212937,0.179114,42.000000,82.000000,"(-7.2122235, -35.906166, 0.55444056)","(-7.2126865, -35.90736, 0.5384905)","[-0.001322385975152172, -0.002644771950304344,...",1.0,"[-0.0017284283257443301, -0.001142642993588310...",1.0,"[-0.0017825553229625494, -0.003565110645925099...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.589147,1.15,0.142438


Index(['minute', 'prn', 's4A', 's4B', 'elev', 'azim', 'rAloc', 'rBloc',
       'auto_cor_A', 'auto_cor_Amax', 'auto_cor_B', 'auto_cor_Bmax',
       'corr_norm', 'lag_norm', 'max_corr', 'time_delay', 'distance (km)'],
      dtype='object')


In [ ]:
plt.plot(cross_cor['lag_norm'], cross_cor['auto_cor_A'])

In [8]:
df = pd.read_parquet('_corrs.pq')
display(df)

,minute,prn,s4A,s4B,elev,azim,rAloc,rBloc,auto_cor_A,auto_cor_Amax,auto_cor_B,auto_cor_Bmax,corr_norm,lag_norm,max_corr,time_delay,distance (km)
0,2022-10-04 22:22:00,E02,0.248260,0.246203,32.000000,306.131250,"[-7.2121882, -35.90611, 0.55649865]","[-7.212652, -35.9073, 0.5425475]","[0.0007473841554558992, 0.0014947683109117983,...",1.0,"[7.453151618398635e-05, 0.000489778534923339, ...",1.0,"[0.00023601625850355374, 0.0015509639844519247...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.724705,0.30,0.141599
1,2022-10-04 22:23:00,E02,0.258300,0.282083,32.000000,307.000000,"[-7.2121882, -35.906105, 0.5563686]","[-7.212652, -35.907295, 0.542493]","[-0.0002862061228668591, -0.000572412245733718...",1.0,"[5.653192470854709e-08, 1.1306384941709418e-07...",1.0,"[-1.7930009852308093e-05, -3.5860019704616186e...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.645983,0.30,0.141591
2,2022-10-04 22:24:00,E02,0.356450,0.339789,31.639413,307.360587,"[-7.2121873, -35.906105, 0.55611974]","[-7.212652, -35.907295, 0.54223895]","[-6.744767325199305e-05, -0.000134895346503986...",1.0,"[1.4328560169239606e-05, 2.8657120338479213e-0...",1.0,"[-0.0006721788745185038, -0.001344357749037007...","[-476, -475, -474, -473, -472, -471, -470, -46...",0.811655,0.30,0.141626
3,2022-10-04 22:25:00,E02,0.316135,0.314274,31.000000,308.000000,"[-7.2121873, -35.906105, 0.55578876]","[-7.212652, -35.907295, 0.5419835]","[-0.00026176338703215663, -0.00052352677406431...",1.0,"[-0.00020526582306656593, -0.00030202107887327...",1.0,"[4.9965221128560714e-05, 9.993044225712143e-05...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.744665,0.30,0.141619
4,2022-10-04 22:26:00,E02,0.345028,0.325862,31.000000,308.000000,"[-7.2121882, -35.906105, 0.5553776]","[-7.212654, -35.907295, 0.5417687]","[-0.0003301509261376589, -0.000660301852275317...",1.0,"[-0.00012863502852089365, 0.000664294326391474...",1.0,"[-0.0001734960252106696, -0.000346992050421339...","[-479, -478, -477, -476, -475, -474, -473, -47...",0.686357,0.35,0.141635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,2022-10-04 23:37:00,S136,0.332755,0.340081,42.000000,82.000000,"[-7.2122126, -35.906174, 0.55261135]","[-7.2126813, -35.907364, 0.53728354]","[-8.878722740604429e-05, -0.000177574454812088...",1.0,"[-0.0015268451462488895, -0.003053690292497779...",1.0,"[-0.0002986801244202465, -0.000597360248840493...","[-949, -948, -947, -946, -945, -944, -943, -94...",0.862867,0.90,0.141914
1531,2022-10-04 23:38:00,S136,0.311478,0.274253,42.000000,82.000000,"[-7.2122173, -35.906174, 0.5526083]","[-7.2126837, -35.907364, 0.5371062]","[0.0005966432587921004, 0.0011932865175842008,...",1.0,"[-0.0020429527221507743, -0.004085905444301549...",1.0,"[-0.001273931181367274, -0.002547862362734548,...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.798184,0.90,0.141864
1532,2022-10-04 23:39:00,S136,0.263936,0.258914,42.000000,82.000000,"[-7.2122197, -35.906174, 0.55342394]","[-7.2126837, -35.907364, 0.53767467]","[-0.0007197609850313849, -0.001439521970062769...",1.0,"[0.0010492216341598936, 0.002098443268319787, ...",1.0,"[-0.000405569010462156, -0.000811138020924312,...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.765527,0.80,0.141787
1533,2022-10-04 23:40:00,S136,0.212937,0.179114,42.000000,82.000000,"[-7.2122235, -35.906166, 0.55444056]","[-7.2126865, -35.90736, 0.5384905]","[-0.001322385975152172, -0.002644771950304344,...",1.0,"[-0.0017284283257443301, -0.001142642993588310...",1.0,"[-0.0017825553229625494, -0.003565110645925099...","[-959, -958, -957, -956, -955, -954, -953, -95...",0.589147,1.15,0.142438
